# 06 · Resource-aware model benchmark for `arrival_pre`

This notebook performs a technical benchmark—not final model selection—for regularised
linear regression, Random Forest, Spark GBT and Spark XGBoost. It uses a deterministic 1%
train sample and 5% validation sample, never reads test, and records time, memory and
segmented error metrics.

Start it with at least 5.5 GB available RAM. Its in-kernel guard allows 5.0 GB after imports
and uses two Spark threads,
a 4 GB driver and an 8,192-position categorical hash. Expected runtime is 25–50 minutes.


In [1]:
from pathlib import Path
import gc
import math
import sys
import threading
import time
import traceback

import pandas as pd
import psutil
from pyspark import StorageLevel
from pyspark.ml.feature import FeatureHasher, VectorAssembler
from pyspark.ml.regression import (
    GBTRegressor, LinearRegression, RandomForestRegressor,
)
from pyspark.sql import functions as F

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.flight_config import SEED
from src.spark_flight_pipeline import (
    apply_yeo_johnson_lambdas, create_spark, fit_yeo_johnson_lambdas,
)

RUN_BENCHMARK = True
RUN_ONLY_MODELS = {"xgboost"}  # Use None for the complete six-run benchmark.
MIN_AVAILABLE_RAM_GB = 5.0  # Checked after Python/PySpark imports (~0.5 GB overhead).
TRAIN_SAMPLE_PERCENT = 1
VALIDATION_SAMPLE_PERCENT = 5
LOW_MEMORY_HASH_FEATURES = 8_192
TARGET = "Arrival_Delay_Min"  # Arrival delay predicted before departure.
DATA_ROOT = PROJECT_ROOT / "data" / "processed" / "model" / "arrival_pre"
REPORT_PATH = PROJECT_ROOT / "reports" / "modeling" / "legacy" / "benchmark" / "06_model_benchmark.csv"
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
EXPECTED_COUNTS = {"train": 2_457_169, "validation": 591_391}
FORBIDDEN_COLUMNS = {
    "ACTUAL OFF BLOCK TIME", "ACTUAL ARRIVAL TIME",
    "Actual Distance Flown (nm)", "Departure_Delay_Min",
}


## Preflight

Before enabling the benchmark: restart Windows if practical; close browsers, Office, Teams,
Docker/WSL and old Jupyter kernels; pause OneDrive sync; connect the laptop to power and use
the best-performance energy profile. No process is terminated automatically.


In [2]:
memory = psutil.virtual_memory()
available_ram_gb = memory.available / (1024 ** 3)
print({
    "run_enabled": RUN_BENCHMARK,
    "logical_cpus": psutil.cpu_count(logical=True),
    "physical_cpus": psutil.cpu_count(logical=False),
    "total_ram_gb": round(memory.total / (1024 ** 3), 2),
    "available_ram_gb": round(available_ram_gb, 2),
    "required_available_ram_gb": MIN_AVAILABLE_RAM_GB,
})
if RUN_BENCHMARK and available_ram_gb < MIN_AVAILABLE_RAM_GB:
    raise RuntimeError(
        f"Only {available_ram_gb:.2f} GB RAM is available; close applications "
        f"and retry with at least {MIN_AVAILABLE_RAM_GB:.1f} GB."
    )


{'run_enabled': True, 'logical_cpus': 20, 'physical_cpus': 14, 'total_ram_gb': 15.63, 'available_ram_gb': 6.76, 'required_available_ram_gb': 5.0}


## Spark session and frozen data contract

Spark starts with four local cores, a requested 6 GB driver heap and 16 shuffle partitions.
The cell rejects an old session whose JVM heap did not adopt the requested memory. Test is
intentionally not loaded.


In [3]:
if RUN_BENCHMARK:
    spark = create_spark(
        "arrival-pre-model-benchmark",
        master="local[2]",
        driver_memory="4g",
        shuffle_partitions=8,
    )
    spark.sparkContext.setLogLevel("WARN")
    jvm_heap_gb = (
        spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / (1024 ** 3)
    )
    if jvm_heap_gb < 3.5:
        spark.stop()
        raise RuntimeError(
            f"Spark JVM heap is only {jvm_heap_gb:.2f} GB. Restart the kernel "
            "so the 4 GB driver-memory setting can take effect."
        )
    train_full = spark.read.parquet(str(DATA_ROOT / "train"))
    validation_full = spark.read.parquet(str(DATA_ROOT / "validation"))
    observed_counts = {
        "train": train_full.count(),
        "validation": validation_full.count(),
    }
    assert observed_counts == EXPECTED_COUNTS, (observed_counts, EXPECTED_COUNTS)
    for name, frame in {"train": train_full, "validation": validation_full}.items():
        assert TARGET in frame.columns, name
        assert not (FORBIDDEN_COLUMNS & set(frame.columns)), name
    assert train_full.schema.json() == validation_full.schema.json()
    print({"jvm_heap_gb": round(jvm_heap_gb, 2), "counts": observed_counts})
else:
    print("Benchmark disabled: set RUN_BENCHMARK=True when ready.")


{'jvm_heap_gb': 4.0, 'counts': {'train': 2457169, 'validation': 591391}}


## Deterministic samples and predeparture temporal features

Sampling uses a stable hash of `ECTRL ID`. Scheduled duration and cyclical time variables
are known before departure. Invalid scheduled durations are imputed with a median learned
from the benchmark train sample only.


In [4]:
LOW_MEMORY_HASH_COLUMN = "high_cardinality_hash_low_memory"
LOW_MEMORY_HASH_INPUTS = ["ADEP", "ADES", "AC Operator", "AC Type_grouped"]
low_memory_hasher = FeatureHasher(
    inputCols=LOW_MEMORY_HASH_INPUTS,
    outputCol=LOW_MEMORY_HASH_COLUMN,
    numFeatures=LOW_MEMORY_HASH_FEATURES,
)

def deterministic_percent_sample(frame, percent):
    return frame.filter(F.pmod(F.hash("ECTRL ID"), F.lit(100)) < percent)

def add_temporal_features(frame):
    duration = (
        F.col("FILED ARRIVAL TIME").cast("long")
        - F.col("FILED OFF BLOCK TIME").cast("long")
    ) / F.lit(60.0)
    hour = F.hour("FILED OFF BLOCK TIME").cast("double")
    day_of_week = F.dayofweek("FILED OFF BLOCK TIME").cast("double") - 1.0
    return (
        frame.withColumn(
            "Scheduled_Duration_Min",
            F.when((duration > 0) & (duration <= 1_440), duration),
        )
        .withColumn("departure_hour_sin", F.sin(2 * F.lit(math.pi) * hour / 24.0))
        .withColumn("departure_hour_cos", F.cos(2 * F.lit(math.pi) * hour / 24.0))
        .withColumn("departure_dow_sin", F.sin(2 * F.lit(math.pi) * day_of_week / 7.0))
        .withColumn("departure_dow_cos", F.cos(2 * F.lit(math.pi) * day_of_week / 7.0))
        .withColumn("departure_month", F.month("FILED OFF BLOCK TIME").cast("double"))
    )

if RUN_BENCHMARK:
    train_sample = low_memory_hasher.transform(add_temporal_features(
        deterministic_percent_sample(train_full, TRAIN_SAMPLE_PERCENT)
    ))
    validation_sample = low_memory_hasher.transform(add_temporal_features(
        deterministic_percent_sample(validation_full, VALIDATION_SAMPLE_PERCENT)
    ))
    duration_median = train_sample.approxQuantile(
        "Scheduled_Duration_Min", [0.5], 0.001
    )[0]
    train_sample = train_sample.fillna({"Scheduled_Duration_Min": duration_median})
    validation_sample = validation_sample.fillna({"Scheduled_Duration_Min": duration_median})
    sample_counts = {
        "train": train_sample.count(),
        "validation": validation_sample.count(),
    }
    print({"sample_counts": sample_counts, "train_duration_median": duration_median})


{'sample_counts': {'train': 24743, 'validation': 29315}, 'train_duration_median': 111.05}


## Numeric variants and one common feature vector

The target always remains `Arrival_Delay_Min` in minutes. Log and Yeo–Johnson replace only
the two continuous predictors. Yeo–Johnson lambdas are estimated from benchmark train and
then reused unchanged for validation. Trees use the original numeric variant initially.


In [5]:
CONTINUOUS_COLUMNS = ["Requested_FL_Imputed", "Scheduled_Duration_Min"]
COMMON_FEATURE_COLUMNS = [
    "departure_hour_sin", "departure_hour_cos",
    "departure_dow_sin", "departure_dow_cos", "departure_month",
    "STATFOR Market Segment_ohe", "Class_aircraft_ohe",
    "Number+Engine Type_aircraft_ohe", LOW_MEMORY_HASH_COLUMN,
]

def add_log_predictors(frame):
    return (
        frame.withColumn("Requested_FL_Imputed_Log1p",
                         F.log1p(F.col("Requested_FL_Imputed")))
        .withColumn("Scheduled_Duration_Min_Log1p",
                    F.log1p(F.col("Scheduled_Duration_Min")))
    )

def assemble_variant(train_frame, validation_frame, variant):
    if variant == "original":
        numeric_columns = CONTINUOUS_COLUMNS
    elif variant == "log":
        train_frame = add_log_predictors(train_frame)
        validation_frame = add_log_predictors(validation_frame)
        numeric_columns = ["Requested_FL_Imputed_Log1p", "Scheduled_Duration_Min_Log1p"]
    elif variant == "yeo_johnson":
        lambdas = fit_yeo_johnson_lambdas(train_frame, columns=CONTINUOUS_COLUMNS)
        outputs = {column: f"{column}_YJ" for column in CONTINUOUS_COLUMNS}
        train_frame = apply_yeo_johnson_lambdas(train_frame, lambdas, outputs)
        validation_frame = apply_yeo_johnson_lambdas(validation_frame, lambdas, outputs)
        numeric_columns = [outputs[column] for column in CONTINUOUS_COLUMNS]
        print({"yeo_johnson_lambdas": lambdas})
    else:
        raise ValueError(variant)
    assembler = VectorAssembler(
        inputCols=[*numeric_columns, *COMMON_FEATURE_COLUMNS],
        outputCol="features",
        handleInvalid="error",
    )
    selected = ["ECTRL ID", TARGET, "features"]
    return (
        assembler.transform(train_frame).select(*selected).persist(StorageLevel.DISK_ONLY),
        assembler.transform(validation_frame).select(*selected).persist(StorageLevel.DISK_ONLY),
    )

prepared = {}
if RUN_BENCHMARK:
    variants_to_prepare = ("original",) if RUN_ONLY_MODELS == {"xgboost"} else (
        "original", "log", "yeo_johnson"
    )
    for variant in variants_to_prepare:
        train_variant, validation_variant = assemble_variant(
            train_sample, validation_sample, variant
        )
        train_rows = train_variant.count()
        validation_rows = validation_variant.count()
        assert train_rows == sample_counts["train"]
        assert validation_rows == sample_counts["validation"]
        feature_dimension = train_variant.first()["features"].size
        prepared[variant] = {
            "train": train_variant, "validation": validation_variant,
            "feature_dimension": feature_dimension,
        }
    print({variant: values["feature_dimension"] for variant, values in prepared.items()})


{'original': 8224}


## Shared segmented metrics and memory monitor


In [6]:
def collect_segment_metrics(predictions):
    errors = (
        predictions.select(TARGET, "prediction")
        .withColumn("absolute_error", F.abs(F.col(TARGET) - F.col("prediction")))
        .withColumn("squared_error", F.pow(F.col(TARGET) - F.col("prediction"), 2))
    )
    exclusive = errors.withColumn(
        "segment",
        F.when(F.col(TARGET) <= 15, "punctual_<=15")
        .when(F.col(TARGET) <= 60, "moderate_15_60")
        .otherwise("severe_>60"),
    )
    segmented = (
        exclusive.unionByName(errors.withColumn("segment", F.lit("all")))
        .unionByName(errors.filter(F.col(TARGET) > 15).withColumn("segment", F.lit("delayed_>15")))
    )
    return [row.asDict() for row in segmented.groupBy("segment").agg(
        F.count("*").alias("rows"),
        F.avg("absolute_error").alias("MAE"),
        F.sqrt(F.avg("squared_error")).alias("RMSE"),
        F.percentile_approx("absolute_error", 0.5, 10_000).alias("median_absolute_error"),
        F.percentile_approx("absolute_error", 0.9, 10_000).alias("p90_absolute_error"),
    ).collect()]

class MemoryMonitor:
    def __init__(self):
        self.stop_event = threading.Event()
        self.minimum_available_gb = float("inf")
        self.thread = threading.Thread(target=self._poll, daemon=True)
    def _poll(self):
        while not self.stop_event.wait(1.0):
            available = psutil.virtual_memory().available / (1024 ** 3)
            self.minimum_available_gb = min(self.minimum_available_gb, available)
    def start(self):
        self.thread.start()
    def stop(self):
        self.stop_event.set()
        self.thread.join(timeout=2.0)
        return self.minimum_available_gb


## Six sequential benchmark runs

XGBoost runs last because it uses Python workers and barrier execution. A model failure is
recorded without discarding successful earlier results. No benchmark model is persisted.


In [7]:
def build_model(model_name):
    if model_name == "linear_regression":
        return LinearRegression(
            featuresCol="features", labelCol=TARGET, predictionCol="prediction",
            regParam=0.01, elasticNetParam=0.5, maxIter=20, standardization=True,
        )
    if model_name == "random_forest":
        return RandomForestRegressor(
            featuresCol="features", labelCol=TARGET, predictionCol="prediction",
            numTrees=20, maxDepth=6, minInstancesPerNode=20,
            featureSubsetStrategy="sqrt", seed=SEED,
        )
    if model_name == "gradient_boosted_trees":
        return GBTRegressor(
            featuresCol="features", labelCol=TARGET, predictionCol="prediction",
            maxIter=20, maxDepth=5, stepSize=0.1, subsamplingRate=0.8, seed=SEED,
        )
    if model_name == "xgboost":
        from xgboost.spark import SparkXGBRegressor
        return SparkXGBRegressor(
            features_col="features", label_col=TARGET, prediction_col="prediction",
            num_workers=1, n_estimators=50, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror",
            eval_metric="mae", device="cpu", force_repartition=True,
            enable_sparse_data_optim=True, missing=0.0,
            random_state=SEED,
        )
    raise ValueError(model_name)

RUN_SPECS = [
    ("linear_regression", "original"),
    ("linear_regression", "log"),
    ("linear_regression", "yeo_johnson"),
    ("random_forest", "original"),
    ("gradient_boosted_trees", "original"),
    ("xgboost", "original"),
]

ACTIVE_RUN_SPECS = [
    spec for spec in RUN_SPECS
    if RUN_ONLY_MODELS is None or spec[0] in RUN_ONLY_MODELS
]
benchmark_results = []
if RUN_BENCHMARK:
    if RUN_ONLY_MODELS and REPORT_PATH.exists():
        previous_results = pd.read_csv(REPORT_PATH)
        previous_results = previous_results[
            ~previous_results["model"].isin(RUN_ONLY_MODELS)
        ]
        benchmark_results.extend(previous_results.to_dict(orient="records"))
        print(f"Preserved {len(previous_results)} previous metric rows.")
    for model_name, variant in ACTIVE_RUN_SPECS:
        print(f"Starting {model_name} / {variant}")
        monitor = MemoryMonitor()
        monitor.start()
        fit_seconds = None
        prediction_seconds = None
        estimator = model = predictions = None
        try:
            estimator = build_model(model_name)
            fit_started = time.perf_counter()
            model = estimator.fit(prepared[variant]["train"])
            fit_seconds = time.perf_counter() - fit_started
            prediction_started = time.perf_counter()
            predictions = model.transform(prepared[variant]["validation"])
            metric_rows = collect_segment_metrics(predictions)
            prediction_seconds = time.perf_counter() - prediction_started
            minimum_available_gb = monitor.stop()
            for metric_row in metric_rows:
                benchmark_results.append({
                    "model": model_name, "numeric_variant": variant,
                    "status": "ok", "error": None,
                    "train_rows": sample_counts["train"],
                    "validation_rows": sample_counts["validation"],
                    "feature_dimension": prepared[variant]["feature_dimension"],
                    "fit_seconds": fit_seconds,
                    "prediction_seconds": prediction_seconds,
                    "minimum_available_ram_gb": minimum_available_gb,
                    **metric_row,
                })
        except Exception as exc:
            minimum_available_gb = monitor.stop()
            benchmark_results.append({
                "model": model_name, "numeric_variant": variant,
                "status": "error", "error": repr(exc),
                "train_rows": sample_counts["train"],
                "validation_rows": sample_counts["validation"],
                "feature_dimension": prepared[variant]["feature_dimension"],
                "fit_seconds": fit_seconds,
                "prediction_seconds": prediction_seconds,
                "minimum_available_ram_gb": minimum_available_gb,
                "segment": "not_scored",
            })
            print(traceback.format_exc())
        finally:
            predictions = model = estimator = None
            spark.catalog.clearCache()
            spark._jvm.java.lang.System.gc()
            gc.collect()


Preserved 25 previous metric rows.
Starting xgboost / original


2026-08-09 15:12:10,689 INFO XGBoost-PySpark: _fit Running xgboost-3.4.0 on 1 workers with
	booster params: {'objective': 'reg:squarederror', 'colsample_bytree': 0.8, 'device': 'cpu', 'eval_metric': 'mae', 'learning_rate': 0.1, 'max_depth': 6, 'random_state': 42, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 50}
	dmatrix_kwargs: {'nthread': 1, 'missing': 0.0}


2026-08-09 15:12:23,487 INFO XGBoost-PySpark: _fit Finished xgboost training!


## Save the benchmark report and release Spark

The report is diagnostic. Test remains unopened, and these sample-trained models must not be
used as final candidates.


In [8]:
if RUN_BENCHMARK:
    benchmark_pd = pd.DataFrame(benchmark_results)
    REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    benchmark_pd.to_csv(REPORT_PATH, index=False)
    display(benchmark_pd.sort_values(["model", "numeric_variant", "segment"]))
    for values in prepared.values():
        values["train"].unpersist()
        values["validation"].unpersist()
    spark.stop()
    print(f"Benchmark report saved to {REPORT_PATH}")
else:
    print(
        "Notebook prepared but not executed. Free RAM, set RUN_BENCHMARK=True, "
        "restart the kernel and run all cells."
    )


,model,numeric_variant,status,error,train_rows,validation_rows,feature_dimension,fit_seconds,prediction_seconds,minimum_available_ram_gb,segment,rows,MAE,RMSE,median_absolute_error,p90_absolute_error
23,gradient_boosted_trees,original,ok,NaN,24743,29315,8224,31.398981,1.468003,2.843472,all,29315.0,10.923941,16.397427,7.933452,22.702355
24,gradient_boosted_trees,original,ok,NaN,24743,29315,8224,31.398981,1.468003,2.843472,delayed_>15,5668.0,24.043483,30.750454,19.641263,41.755045
22,gradient_boosted_trees,original,ok,NaN,24743,29315,8224,31.398981,1.468003,2.843472,moderate_15_60,5343.0,20.807385,23.447510,18.968021,35.702410
20,gradient_boosted_trees,original,ok,NaN,24743,29315,8224,31.398981,1.468003,2.843472,punctual_<=15,23647.0,7.779290,10.328234,6.399646,15.525938
21,gradient_boosted_trees,original,ok,NaN,24743,29315,8224,31.398981,1.468003,2.843472,severe_>60,325.0,77.244942,86.328553,67.751477,120.070291
8,linear_regression,log,ok,NaN,24743,29315,8224,4.382236,1.461303,3.945908,all,29315.0,10.875577,15.926486,8.215018,21.934566
9,linear_regression,log,ok,NaN,24743,29315,8224,4.382236,1.461303,3.945908,delayed_>15,5668.0,19.322786,27.295365,14.820095,37.502742
7,linear_regression,log,ok,NaN,24743,29315,8224,4.382236,1.461303,3.945908,moderate_15_60,5343.0,16.078862,19.442860,14.099424,31.110523
5,linear_regression,log,ok,NaN,24743,29315,8224,4.382236,1.461303,3.945908,punctual_<=15,23647.0,8.850847,11.656418,7.100678,18.374979
6,linear_regression,log,ok,NaN,24743,29315,8224,4.382236,1.461303,3.945908,severe_>60,325.0,72.652898,82.332966,62.912948,112.813560


Benchmark report saved to C:\Users\celti\OneDrive - Universidade de Santiago de Compostela\Verano\ML_flights_project\reports\06_model_benchmark.csv


## Acceptance criteria for notebook 07

Proceed to 10–20% tuning only if all four engines finish, XGBoost works with Spark 4.2 on
Windows, feature dimensions and row counts are stable, the JVM keeps its 6 GB heap and no
run exhausts available RAM. Final selection will use full validation and must beat the frozen
route+airline baseline MAE of 9.953 minutes.
